# 数据分析

## 1：数据加载与收敛趋势绘图（Regret & RMSE）

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import os
import json
import glob

# 1. 设置路径
results_dir = './results/mc_res/'
fig_save_dir = './results/figs/'
os.makedirs(fig_save_dir, exist_ok=True) # 自动创建 figs 目录

# 2. 搜寻 JSON 碎片
json_files = glob.glob(os.path.join(results_dir, "*.json"))

if len(json_files) > 0:
    all_data = []
    for f in json_files:
        try:
            with open(f, 'r') as jfile:
                all_data.append(json.load(jfile))
        except Exception as e:
            print(f"警告：读取 {f} 出错: {e}")
            
    df = pd.DataFrame(all_data).sort_values(by=['model_type', 'phi_type', 'n_train'])
    
    # 3. 绘图与保存
    plt.figure(figsize=(15, 6))
    
    # 子图 1: True Regret
    plt.subplot(1, 2, 1)
    sns.lineplot(data=df, x='n_train', y='regret', hue='phi_type', style='model_type', 
                 markers=True, palette='tab10', linewidth=2)
    plt.title('True Regret Convergence Trend', fontsize=12, fontweight='bold')
    plt.xlabel('Training Set size (n)', fontsize=10)
    plt.ylabel('True Regret (Oracle - Model)', fontsize=10)
    plt.grid(True, linestyle='--', alpha=0.6)
    
    # 子图 2: RMSE
    plt.subplot(1, 2, 2)
    sns.lineplot(data=df, x='n_train', y='rmse', hue='phi_type', style='model_type', 
                 markers=True, palette='Set1', linewidth=2)
    plt.title('RMSE Convergence Trend', fontsize=12, fontweight='bold')
    plt.xlabel('Training Set size (n)', fontsize=10)
    plt.ylabel('Root Mean Squared Error', fontsize=10)
    plt.grid(True, linestyle='--', alpha=0.6)
    
    plt.tight_layout()
    
    # 4. 执行保存 (保存为高分辨率 PNG 和 PDF 矢量图)
    save_path = os.path.join(fig_save_dir, 'convergence_analysis.png')
    plt.savefig(save_path, dpi=300, bbox_inches='tight')
    plt.savefig(save_path.replace('.png', '.pdf'), bbox_inches='tight')
    
    plt.show()
    print(f"🎨 图像已成功渲染并保存至：\n   - {save_path}\n   - {save_path.replace('.png', '.pdf')} (矢量图)")
else:
    print(f"未在 {results_dir} 发现任何有效 JSON 数据。")


In [ ]:
## 代码块 1：数据加载与收敛趋势绘图（Regret & RMSE）

import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import os

# 加载数据
res_path = './results/mc_res/full_summary.csv'
if os.path.exists(res_path):
    df = pd.read_csv(res_path)
    
    # 绘制 True Regret 随 n 增加的收敛曲线
    plt.figure(figsize=(12, 5))
    plt.subplot(1, 2, 1)
    sns.lineplot(data=df, x='n_train', y='regret', hue='phi_type', style='model_type', markers=True)
    plt.title('True Regret Convergence')
    plt.grid(True)
    
    # 绘制 RMSE 随 n 增加的收敛曲线
    plt.subplot(1, 2, 2)
    sns.lineplot(data=df, x='n_train', y='rmse', hue='phi_type', style='model_type', markers=True)
    plt.title('RMSE Convergence')
    plt.grid(True)
    plt.show()
else:
    print("请先在终端运行 python -m Main.run_parameter_analysis 以生成汇总数据！")


## 2. 生成 $n=10000$ 的表格

In [ ]:
if os.path.exists(res_path):
    # 筛选最大样本量
    df_final = df[df['n_train'] == 10000].copy()
    
    # 整理表格列
    report_table = df_final[['model_type', 'phi_type', 'oracle_q', 'mean_q', 'std_q', 'regret', 'rmse']]
    report_table.columns = ['Model', 'Phi', 'Oracle', 'Mean Q', 'SD', 'Regret', 'RMSE']
    
    # 打印漂亮的美化表格
    print("=== Table: Parameter Analysis Results (n=10000) ===")
    display(report_table.sort_values(by=['Model', 'Phi']).reset_index(drop=True))


In [ ]:
import torch

a = torch.Tensor([[1,2,3],[4,5,6]]).reshape(3,2)
b = torch.Tensor([2,4,5]).unsqueeze(-1)
c = torch.cat([a,b],dim=1)
c 

In [1]:
import numpy as np
from scipy.stats import norm
from scipy.optimize import bisect

# ==========================================
# 第一步：大样本蒙特卡洛初始化
# ==========================================
np.random.seed(2026)  # 设置随机种子，保证每次运行结果一致
N = int(1e4)       # 生成 100 万个虚拟患者样本

# 根据论文 Case 1 设定，X1 和 X2 独立且服从标准正态分布
X1 = np.random.normal(0, 1, N)
X2 = np.random.normal(0, 1, N)

# ==========================================
# 第二步：定义目标函数 F(q)
# ==========================================
def F(q):
    """
    计算给定阈值 q 下，采取最优策略时的平均生存概率。
    对应公式：E[ max( P(Y(1)>q|X), P(Y(0)>q|X) ) ]
    """
    # 1. 计算 A=1 时的条件生存概率: 1 - Phi((q - 2 + X1 - 3*X2) / 3)
    prob_1 = 1 - norm.cdf((q - 2 + X1 - 3 * X2) / 3.0)
    
    # 2. 计算 A=0 时的条件生存概率: 1 - Phi(q - 2 + 2*X1 - 2*X2)
    prob_0 = 1 - norm.cdf(q - 2 + 2 * X1 - 2 * X2)
    
    # 3. 对每个虚拟患者，挑选能带来最大生存概率的治疗方案
    max_prob = np.maximum(prob_1, prob_0)
    
    # 4. 求全体样本的平均值（蒙特卡洛积分近似期望）
    return np.mean(max_prob)

# ==========================================
# 第三步：定义二分法寻根方程
# ==========================================
def objective(q, tau):
    """
    我们要找的 q^* 满足 F(q) = 1 - tau
    因此目标寻根方程为: F(q) - (1 - tau) = 0
    """
    return F(q) - (1 - tau)

# ==========================================
# 第四步：求解不同 tau 下的最优分位数 q^*
# ==========================================
print("开始执行大样本蒙特卡洛寻根计算...\n")

# 1. 当 tau = 0.25 时 (寻找使 F(q) = 0.75 的 q)
# bisect 函数在区间 [-5, 5] 内进行二分查找
q_25_optimal = bisect(objective, -3, 3, args=(0.25,))
print(f"当 tau = 0.25 时：")
print(f" -> 程序求解得到的 q^* = {q_25_optimal:.6f}")
print(f" -> 论文 Table 1 报告值 = 0.485\n")

# 2. 当 tau = 0.50 时 (寻找使 F(q) = 0.50 的 q)
# bisect 函数在区间 [-5, 10] 内进行二分查找
q_50_optimal = bisect(objective, -10, 10, args=(0.50,))
print(f"当 tau = 0.50 时：")
print(f" -> 程序求解得到的 q^* = {q_50_optimal:.6f}")
print(f" -> 论文 Table 1 报告值 = 2.789\n")

开始执行大样本蒙特卡洛寻根计算...

当 tau = 0.25 时：
 -> 程序求解得到的 q^* = 0.473539
 -> 论文 Table 1 报告值 = 0.485

当 tau = 0.50 时：
 -> 程序求解得到的 q^* = 2.819750
 -> 论文 Table 1 报告值 = 2.789

